# SHA-256 Implementation

This notebook works through the problems based on the [Secure Hash Standard](https://csrc.nist.gov/pubs/fips/180-4/upd1/final), building up SHA-256 in Python. Each problem has markdown explanations with the code and tests below them.

In [1]:
# Numerical arrays.
import numpy as np

## Problem 0: GitHub Issues

I created one GitHub issue per problem to track my progress, and commented on each one as I worked. They are on the [repository's issues page](https://github.com/MarkJohnBowman/computational-theory/issues).

## Problem 1: Representing SHA-256 Data

### 32-bit words

FIPS 180-4 section 2.1 defines a **word** as "a group of either 32 bits or 64 bits, depending on the algorithm. For SHA-256 specifically, section 3.2 states the algorithm operates on 32-bit words.

Source: [FIPS 180-4 - Secure Hash Standard](https://csrc.nist.gov/pubs/fips/180-4/upd1/final)

In [2]:
# The largest value a 32-bit word can hold, per FIPS 180-4 Sec 3.1(3)
max_32_bit = 2**32 -1

# Show value
print("Max 32-bit value:", max_32_bit)

Max 32-bit value: 4294967295


A normal Python `int` has no fixed size, so if I add 1 to the biggest 32-bit value it just becomes a bigger number.

In [3]:
# A Python int has no fixed size, so adding 1 just gives a bigger number
python_int = max_32_bit + 1

# Show result
print("Python int:", python_int)

Python int: 4294967296


A NumPy `uint32` is a real 32-bit word, so it can't go past its maximum. Adding 1 wraps round to 0. NumPy gives a warning when this happens, so I use `np.errstate` to turn the warning off for this line.

In [4]:
# A uint32 is a real 32-bit word, so adding 1 wraps round to 0
with np.errstate(over="ignore"): numpy_word = np.uint32(max_32_bit) + np.uint32(1)

# Show result
print("numpy.uint32:", numpy_word)

numpy.uint32: 0


### Sequences of 32-bit words

A few parts of SHA-256 need more than just one word - a message block consists of sixteen words. For this we must find a way to put words together in a sequence.

From my research, a NumPy array seemed the best fit over a plain list. Every value in the array is forced to be a 32-bit word, which matches how the standard treats every value. It is also easy to apply an operation to the whole sequence in this format.

- [How to create an array and specify its dtype](https://numpy.org/doc/stable/reference/generated/numpy.array.html)

- [NumPy wraparound/overflow behaviour](https://numpy.org/doc/stable/user/basics.types.html)


In [5]:
# a sequence of 32-bit words
word_sequence= np.array([1,2,3,4], dtype=np.uint32)

# show the array and its type
print(word_sequence)
print(word_sequence.dtype)


[1 2 3 4]
uint32


To check the array still acts like 32-bit words, I add the biggest 32-bit value to every item. Each one should wrap round, so 1 becomes 0, 2 becomes 1, and so on.

In [6]:
# checking the whole array still wraps around like a 32-bit word
print(word_sequence +np.uint32(2**32 -1))

[0 1 2 3]


### Input messages

FIPS 180-4 works on bits and bytes, not text. In python, a string represents Unicode text, not raw bytes, so it can't be hashed directly, it needs to be converted first.

I will represent the input message as a Python bytes object. If the message starts out as a string, I can convert it using .encode(), which turns it into its raw byte representation (UTF-8 by default).

- [Python docs - Text vs Data](https://docs.python.org/3/library/stdtypes.html#text-sequence-type-str)

In [7]:
# a string is text, not raw bytes - can't be hashed directly 
message = "abc"

# show type
print(type(message))

<class 'str'>


I convert the string to bytes with `.encode("utf-8")`. This gives a `bytes` object, which is what SHA-256 needs.

In [8]:
# converting to bytes using UTF-8 encoding
message_bytes = message.encode("utf-8")

# show bytes and type
print(message_bytes)
print(type(message_bytes))

b'abc'
<class 'bytes'>


Some characters take more than one byte once they're encoded, so the number of characters and the number of bytes can be different. I tested this with "café".

In [9]:
# non-ASCII chracters can take more than one byte once encoded
message2 = "café"

print(len(message2))  #character count
print(len(message2.encode("utf-8")))   #byte count - different

4
5


"café" has 4 characters but 5 bytes, because é takes two bytes in UTF-8. The padding needs the length in bits, so the length has to be counted in bytes, not characters.

### 512-bit message blocks

[FIPS 180-4 section 3.1(4a)](https://csrc.nist.gov/pubs/fips/180-4/upd1/final) defines a SHA-256 message block as 512 bits, represented as sixteen 32-bit words.

This works because a 64-byte slice of the message (512 / 8 = 64 bytes) is the right size, and splitting it into sixteen 4-byte groups gives the sixteen words. I use [slicing](https://docs.python.org/3/library/stdtypes.html#common-sequence-operations) to take the block and [int.from_bytes()](https://docs.python.org/3/library/stdtypes.html#int.from_bytes) to read each group as a big-endian integer.

In [10]:
# 64 bytes of distinct values (0-63), so each word is easy to verify by eye
block_message = bytes(range(64))
print(block_message)
print(len(block_message), "bytes - exactly one 512-bit block")

b'\x00\x01\x02\x03\x04\x05\x06\x07\x08\t\n\x0b\x0c\r\x0e\x0f\x10\x11\x12\x13\x14\x15\x16\x17\x18\x19\x1a\x1b\x1c\x1d\x1e\x1f !"#$%&\'()*+,-./0123456789:;<=>?'
64 bytes - exactly one 512-bit block


The message is exactly 64 bytes, so it makes one full block. I take it with a slice, `[0:64]`, which gives the first 64 bytes.

In [11]:
# slicing out the first 64-byte block
block = block_message[0:64]
print(len(block_message), "bytes")

64 bytes


The words are built with a list comprehension, which creates a list by going through a sequence one item at a time. I used [Real Python's guide to list comprehensions](https://realpython.com/list-comprehension-python/) to understand how they work. The Python tutorial also has a section on [list comprehensions](https://docs.python.org/3/tutorial/datastructures.html#list-comprehensions).

Here, `range(0, 64, 4)` gives the start of each 4-byte group (0, 4, 8, and so on up to 60), `block[i:i + 4]` slices out that group, and `int.from_bytes` reads it as one big-endian integer. Doing this for all sixteen groups gives sixteen words.

In [12]:
# The start position of each 4-byte group in the block
print(list(range(0, 64, 4)))

# The first group of four bytes
print(block[0:4])

# The same four bytes read as one big-endian integer
print(int.from_bytes(block[0:4], byteorder="big"))

[0, 4, 8, 12, 16, 20, 24, 28, 32, 36, 40, 44, 48, 52, 56, 60]
b'\x00\x01\x02\x03'
66051


Now I do the same thing for all sixteen groups at once. I put the results in a `np.uint32` array so each word is a 32-bit unsigned integer.

In [13]:
# split the 64-byte block into sixteen 4-byte groups, one word each
words = np.array(
    [int.from_bytes(block[i:i + 4], byteorder="big") for i in range(0, 64, 4)],
    dtype=np.uint32,
)

# show words, how many and type
print(words)
print(len(words), "words")
print(words.dtype)

[     66051   67438087  134810123  202182159  269554195  336926231
  404298267  471670303  539042339  606414375  673786411  741158447
  808530483  875902519  943274555 1010646591]
16 words
uint32


The first word printed for `words` is 66051, which matches a manual check: bytes 0-3 are 0x00, 0x01, 0x02, 0x03, and read big-endian that is 0x00010203 = 66051. So each group of four bytes was read as one big-endian 32-bit word, and the array holds sixteen of them, as the standard says a block should.

### Big-endian integers

[FIPS 180-4 section 3.1(2)](https://csrc.nist.gov/pubs/fips/180-4/upd1/final) 
says the standard uses "the 'big-endian' convention... so that within each 
word, the most significant bit is stored in the left-most bit position."

Basically, endianness is about which end of a group of bytes you treat as 
the "start" when reading it as one number - and the same bytes can give 
two completely different values depending on which way you read them. 
Python's `int.from_bytes()` has a `byteorder` argument for exactly this, 
which explains why I've been writing `byteorder="big"` in my earlier cells 



In [14]:
# the same four bytes, interpreted two different ways
sample_bytes = bytes([0x00, 0x01, 0x02, 0x03])

# read with most significant byte first
big_endian_value = int.from_bytes(sample_bytes, byteorder="big")

# read with least significant byte first
little_endian_value = int.from_bytes(sample_bytes, byteorder="little")

# results
print("Big-endian:", big_endian_value)
print("Little-endian:", little_endian_value)

Big-endian: 66051
Little-endian: 50462976


The standard gives its own example: 291 written in hex is `00000123`. I read those bytes as big-endian to check I get 291 back. The `assert` stops with an error if the answer is wrong.

In [15]:
# FIPS 180-4 section 3.1(3) gives 291 as an example, represented in hex as "00000123"
fips_example_bytes = bytes.fromhex("00000123")

# read bytes as big endian integer
result = int.from_bytes(fips_example_bytes, byteorder="big")

print(result)  # should be 291, matching the standard's own example
assert result == 291

291


## Problem 2: SHA-256 Bitwise Operations

Before writing these functions I watched [SHA-256 | COMPLETE Step-By-Step Explanation (W/ Example)](https://www.youtube.com/watch?v=orIgy2MjqrA&t=409s) by RedBlockBlue, which goes through SHA-256 step by step. It helped me see where these six functions fit into the algorithm before I tried to write them.

### Ch(x, y, z)

[FIPS 180-4 equation 4.2](https://csrc.nist.gov/pubs/fips/180-4/upd1/final) 
defines this function as:

Ch(x, y, z) = (x AND y) XOR (NOT x AND z)

x controls which of the other two values gets 
picked, one bit at a time. If a bit in x is 1, the matching bit from y is 
used. If a bit in x is 0, the matching bit from z is used instead. This happens separately for all 32 bits at once.

I implemented this directly using Python's bitwise operators - `&` for 
AND, `^` for XOR, and `~` for NOT. These are 
[the same operators NumPy uses](https://numpy.org/doc/stable/reference/routines.bitwise.html), 
and they work on each bit of the number individually, which is exactly what this function needs.

In [16]:
def ch(x, y, z):
    """
    SHA-256 Ch (choose) function.

    For each bit position, selects the bit from y where x is 1,
    and the bit from z where x is 0.

    Parameters
    ----------
    x, y, z : numpy.uint32
        32-bit unsigned integer inputs.

    Returns
    -------
    numpy.uint32
        Result of (x AND y) XOR (NOT x AND z).
    """

    # Take the bit from y where x is 1, and from z where x is 0
    return (x & y) ^ (~x & z)

For the first test, I check the easy cases. When x is all 1s, Ch should just give me y, and when x is all 0s, it should give me z.

In [17]:
# All 1s and all 0s for x, and two different values for y and z
x_all_ones = np.uint(0xFFFFFFFF)
z_all_zeros = np.uint(0x00000000)
y_test = np.uint(0x12345678)
z_test = np.uint(0x87654321)

# With x all 1s, the result should be y
assert ch(x_all_ones, y_test, z_test) == y_test

# With x all 0s, the result should be z
assert ch(z_all_zeros, y_test, z_test) == z_test

# Reached only if both checks pass
print("Identity tests passed")

Identity tests passed


For the second test, I mix the bits and worked out the answer by hand. The lowest four bits of x are 1, so those come from y, which is 0 there. All the other bits of x are 0, so they come from z. That gives 0xA0.

In [18]:
# x is 1 in the lowest four bits only, so those bits come from y
x_mixed = np.uint32(0x0000000F)
y_mixed = np.uint32(0x000000F0)
z_mixed = np.uint32(0x000000AA)

# Worked out by hand, the answer should be 0xA0
result = ch(x_mixed, y_mixed, z_mixed)
print(hex(result))
assert result == np.uint32(0xA0)

0xa0


### Maj(x, y, z)

[FIPS 180-4 equation 4.3](https://csrc.nist.gov/pubs/fips/180-4/upd1/final) 
defines this function as:

Maj(x, y, z) = (x AND y) XOR (x AND z) XOR (y AND z)

At each bit position, this looks at the three 
matching bits from x, y and z, and picks whichever value shows up most - 
a majority vote done separately for every single bit. If two or more of 
the three bits are 1, the output bit is 1. Otherwise it's 0.

I implemented this directly using `&` for AND and `^` for XOR, the same 
[bitwise operators](https://numpy.org/doc/stable/reference/routines.bitwise.html) 
I used for Ch.

In [19]:
def maj(x, y, z):
    """
    SHA-256 Maj (majority) function.

    For each bit position, returns the majority value among
    x, y, and z (the bit that appears in at least two of the three).

    Parameters
    ----------
    x, y, z : numpy.uint32
        32-bit unsigned integer inputs.

    Returns
    -------
    numpy.uint32
        Result of (x AND y) XOR (x AND z) XOR (y AND z).
    """

     # Each bit is 1 if at least two of x, y and z have a 1 there
    return (x & y) ^ (x & z) ^ (y & z)

For the first test, I use the same value for x, y and z. Every bit then has all three the same, so Maj should just give that value back.

In [20]:
# The same value for x, y and z, so every bit is a 3 to 0 majority
same_value = np.uint32(0x5A5A5A5A)

# The result should be that same value
assert maj(same_value, same_value, same_value) == same_value
print("All-equal test passed")

All-equal test passed


For the second test, I use the same inputs as the Ch test. In the lowest 8 bits, x and y are opposites (one is 1 where the other is 0), so z decides each bit. That means the answer is z, which is 0xAA.

In [21]:
# The same inputs as the ch test, so the two functions can be compared
x_mixed = np.uint32(0x0000000F)
y_mixed = np.uint32(0x000000F0)
z_mixed = np.uint32(0x000000AA)

# Worked out by hand, the majority at each bit gives 0xAA
result = maj(x_mixed, y_mixed, z_mixed)
print(hex(result))
assert result == np.uint32(0xAA)

0xaa


### ROTR (rotate right)

[FIPS 180-4 section 3.2](https://csrc.nist.gov/pubs/fips/180-4/upd1/final) 
defines this operation as:

ROTR^n(x) = (x >> n) OR (x << (32 - n))

Every bit in x moves n places to the right. 
Any bit that falls off the right end wraps around and comes back in at 
the left end, so no bits are lost. This is also called a 
[circular shift](https://en.wikipedia.org/wiki/Circular_shift). Unlike a 
normal shift, the empty spaces are filled with the bits that fell off 
instead of zeros. 
[Baeldung's guide to bit shift operators](https://www.baeldung.com/cs/bitwise-shift-operators) 
explains the same difference, and a 
[paper on bit rotation](https://jesit.springeropen.com/articles/10.1186/s43067-021-00029-8) 
mentions that rotations are used in cryptography, which fits with SHA-256 
using them.

I implemented this directly using the 
[right shift and left shift operators](https://numpy.org/doc/stable/reference/generated/numpy.right_shift.html) 
(`>>` and `<<`) and `|` for OR. Because the values are uint32, bits shifted 
off the left end are dropped, which is what the formula relies on.

In [22]:
def rotr(x, n):
    """
    Rotate a 32-bit word right by n positions (FIPS 180-4 section 3.2).

    Parameters
    ----------
    x : numpy.uint32
        The 32-bit word to rotate.
    n : int
        Number of positions to rotate, 0 < n < 32.

    Returns
    -------
    numpy.uint32
        x rotated right by n bits.
    """

    # Convert n so NumPy keeps everything as 32-bit values
    n = np.uint32(n)

    # Shift right, then bring the bits that fell off round to the left
    return (x >> n) | (x << (np.uint32(32) - n))

To test `rotr`, I rotate a single 1 right by 1. It should wrap round to the front and give 0x80000000. I also rotate 0x12345678 right by 8, which moves the last byte (78) round to the front and gives 0x78123456.

In [23]:
# A single 1 in the last place rotates round to the very front
assert rotr(np.uint32(0x00000001), 1) == np.uint32(0x80000000)

# The last byte, 0x78, rotates round to the front
assert rotr(np.uint32(0x12345678), 8) == np.uint32(0x78123456)
print("rotr tests passed")

rotr tests passed


### SHR (shift right)

[FIPS 180-4 section 3.2](https://csrc.nist.gov/pubs/fips/180-4/upd1/final) 
defines this operation as:

SHR^n(x) = x >> n

Every bit in x moves n places to the right, 
the same as ROTR. The difference is that bits falling off the right end 
are thrown away, and zeros fill in on the left. This is known as a 
[logical shift](https://en.wikipedia.org/wiki/Logical_shift), and 
[Baeldung](https://www.baeldung.com/cs/bitwise-shift-operators) describes 
it the same way, padding the empty places with zeros.

I implemented this directly using the `>>` operator, which does exactly 
this on a uint32.

In [24]:
def shr(x, n):
    """
    Shift a 32-bit word right by n positions (FIPS 180-4 section 3.2).

    Parameters
    ----------
    x : numpy.uint32
        The 32-bit word to shift.
    n : int
        Number of positions to shift, 0 <= n < 32.

    Returns
    -------
    numpy.uint32
        x shifted right by n bits, with zeros filled in on the left.
    """

    # Shift right, so the bits that fall off the end are lost
    return x >> np.uint32(n)

To test `shr`, I shift a single 1 right by 1. It falls off the end, so the answer is 0. I also shift 0x0F right by 1, which turns 1111 into 0111, or 0x07.

In [25]:
# The single 1 falls off the end, so the result is 0
assert shr(np.uint32(0x00000001), 1) == np.uint32(0x00000000)

# 1111 shifted right by 1 gives 0111
assert shr(np.uint32(0x0000000F), 1) == np.uint32(0x00000007)
print("shr tests passed")

shr tests passed


### Sigma0 (Σ0)

[FIPS 180-4 equation 4.4](https://csrc.nist.gov/pubs/fips/180-4/upd1/final) 
defines this function as:

Σ0(x) = ROTR^2(x) XOR ROTR^13(x) XOR ROTR^22(x)

It takes one 32-bit word, makes three copies, 
rotates them right by 2, 13 and 22 places, and XORs the three results 
together. The numbers 2, 13 and 22 are fixed by the standard. This mixes 
the bits of the word around so each output bit depends on three different 
input bits.

I implemented this directly by calling the `rotr` function from above three 
times and combining the results with the 
[bitwise XOR operator](https://numpy.org/doc/stable/reference/generated/numpy.bitwise_xor.html) 
(`^`). I called the function `big_sigma0` because Python function names are 
conventionally lowercase, and it needs to be different from the lowercase 
σ0 function later on.

In [26]:
def big_sigma0(x):
    """
    SHA-256 Sigma0 function, written as capital sigma 0 in the standard.

    XORs together x rotated right by 2, 13 and 22 positions
    (FIPS 180-4 equation 4.4).

    Parameters
    ----------
    x : numpy.uint32
        32-bit unsigned integer input.

    Returns
    -------
    numpy.uint32
        Result of ROTR(x, 2) XOR ROTR(x, 13) XOR ROTR(x, 22).
    """
    # Rotate right by 2, 13 and 22, then XOR the three results
    return rotr(x, 2) ^ rotr(x, 13) ^ rotr(x, 22)

The cell below shows the three rotated copies for an input of 1. Each rotation just moves the single 1 bit to a new place.

In [27]:
# A single 1 bit in the last place, so each rotation just moves it
x = np.uint32(0x00000001)

# Show x, each rotated copy and the XOR of the three, all in hex
print(f"x            = {int(x):08x}")
print(f"ROTR 2       = {int(rotr(x, 2)):08x}")
print(f"ROTR 13      = {int(rotr(x, 13)):08x}")
print(f"ROTR 22      = {int(rotr(x, 22)):08x}")
print(f"XOR of three = {int(big_sigma0(x)):08x}")

x            = 00000001
ROTR 2       = 40000000
ROTR 13      = 00080000
ROTR 22      = 00000400
XOR of three = 40080400


Now the tests. For an input of 1, the three rotated copies above are 40000000, 00080000 and 00000400. XORing them gives 40080400. For all 1s, every rotation is still all 1s, and XORing three of them leaves one copy, so the answer is all 1s.

In [28]:
# A single 1 bit, with the answer worked out by hand
result = big_sigma0(np.uint32(0x00000001))
expected = np.uint32(0x40080400)
print(f"result   = {int(result):08x}")
print(f"expected = {int(expected):08x}")
assert result == expected

# All 1s rotates to all 1s, and three copies XORed leave one copy
result = big_sigma0(np.uint32(0xFFFFFFFF))
expected = np.uint32(0xFFFFFFFF)
print(f"result   = {int(result):08x}")
print(f"expected = {int(expected):08x}")
assert result == expected

# Reached only if both checks pass
print("big_sigma0 tests passed")

result   = 40080400
expected = 40080400
result   = ffffffff
expected = ffffffff
big_sigma0 tests passed


### Sigma1 (Σ1)

[FIPS 180-4 equation 4.5](https://csrc.nist.gov/pubs/fips/180-4/upd1/final) 
defines this function as:

Σ1(x) = ROTR^6(x) XOR ROTR^11(x) XOR ROTR^25(x)

It works the same way as Σ0, but with different rotation amounts. It takes one 32-bit word, rotates it right by 6, 11 and 25 
places, and XORs the three results together. The numbers 6, 11 and 25 are 
fixed by the standard.

I implemented this directly by calling the `rotr` function three times and 
combining the results with the 
[bitwise XOR operator](https://numpy.org/doc/stable/reference/generated/numpy.bitwise_xor.html) 
(`^`). I called the function `big_sigma1` to match `big_sigma0`.

In [29]:
def big_sigma1(x):
    """
    SHA-256 Sigma1 function, written as capital sigma 1 in the standard.

    XORs together x rotated right by 6, 11 and 25 positions
    (FIPS 180-4 equation 4.5).

    Parameters
    ----------
    x : numpy.uint32
        32-bit unsigned integer input.

    Returns
    -------
    numpy.uint32
        Result of ROTR(x, 6) XOR ROTR(x, 11) XOR ROTR(x, 25).
    """

     # Rotate right by 6, 11 and 25, then XOR the three results
    return rotr(x, 6) ^ rotr(x, 11) ^ rotr(x, 25)

The cell below shows the three rotated copies for an input of 1. Each rotation just moves the single 1 bit to a new place.

In [30]:
# A single 1 bit in the last place, so each rotation just moves it
x = np.uint32(0x00000001)

# Show x, each rotated copy and the XOR of the three, all in hex
print(f"x            = {int(x):08x}")
print(f"ROTR 6       = {int(rotr(x, 6)):08x}")
print(f"ROTR 11      = {int(rotr(x, 11)):08x}")
print(f"ROTR 25      = {int(rotr(x, 25)):08x}")
print(f"XOR of three = {int(big_sigma1(x)):08x}")

x            = 00000001
ROTR 6       = 04000000
ROTR 11      = 00200000
ROTR 25      = 00000080
XOR of three = 04200080


These are the same tests as Σ0. For an input of 1, the three copies above are 04000000, 00200000 and 00000080. XORing them gives 04200080. All 1s gives all 1s again.

In [31]:
# A single 1 bit, with the answer worked out by hand
result = big_sigma1(np.uint32(0x00000001))
expected = np.uint32(0x04200080)
print(f"result   = {int(result):08x}")
print(f"expected = {int(expected):08x}")
assert result == expected

# All 1s rotates to all 1s, and three copies XORed leave one copy
result = big_sigma1(np.uint32(0xFFFFFFFF))
expected = np.uint32(0xFFFFFFFF)
print(f"result   = {int(result):08x}")
print(f"expected = {int(expected):08x}")
assert result == expected

# Reached only if both checks pass
print("big_sigma1 tests passed")

result   = 04200080
expected = 04200080
result   = ffffffff
expected = ffffffff
big_sigma1 tests passed


### sigma0 (σ0)

[FIPS 180-4 equation 4.6](https://csrc.nist.gov/pubs/fips/180-4/upd1/final) 
defines this function as:

σ0(x) = ROTR^7(x) XOR ROTR^18(x) XOR SHR^3(x)

It takes one 32-bit word and combines three 
copies of it. Two copies are rotated right by 7 and 18 places, and the 
third is shifted right by 3 places. The three results are XORed together. 
The difference from Σ0 is the last term, which is a shift instead of a 
rotation, so the 3 bits that fall off the end are lost instead of wrapping 
round.

I implemented this directly by calling the `rotr` and `shr` functions from 
above and combining the results with the 
[bitwise XOR operator](https://numpy.org/doc/stable/reference/generated/numpy.bitwise_xor.html) 
(`^`). I called the function `small_sigma0` to go with `big_sigma0`.

In [32]:
def small_sigma0(x):
    """
    SHA-256 sigma0 function, written as lowercase sigma 0 in the standard.

    XORs together x rotated right by 7 and 18 positions, and x shifted
    right by 3 positions (FIPS 180-4 equation 4.6).

    Parameters
    ----------
    x : numpy.uint32
        32-bit unsigned integer input.

    Returns
    -------
    numpy.uint32
        Result of ROTR(x, 7) XOR ROTR(x, 18) XOR SHR(x, 3).
    """
    # Rotate right by 7 and 18, shift right by 3, then XOR the three results
    return rotr(x, 7) ^ rotr(x, 18) ^ shr(x, 3)

The cell below shows the three copies for an input of 0x80000000. I used this instead of 1 because shifting 1 right just gives 0, so you couldn't see the shift working.

In [33]:
# A single 1 bit in the top place, so the shift shows up
x = np.uint32(0x80000000)

# Show x, each rotated or shifted copy and the XOR of the three, all in hex
print(f"x            = {int(x):08x}")
print(f"ROTR 7       = {int(rotr(x, 7)):08x}")
print(f"ROTR 18      = {int(rotr(x, 18)):08x}")
print(f"SHR 3        = {int(shr(x, 3)):08x}")
print(f"XOR of three = {int(small_sigma0(x)):08x}")

x            = 80000000
ROTR 7       = 01000000
ROTR 18      = 00002000
SHR 3        = 10000000
XOR of three = 11002000


For an input of 0x80000000, the three copies above are 01000000, 00002000 and 10000000. XORing them gives 11002000. For all 1s, the two rotated copies are both all 1s, so they cancel out when XORed. That leaves the shifted copy, which is 1FFFFFFF.

In [34]:
# A single 1 bit in the top place, with the answer worked out by hand
result = small_sigma0(np.uint32(0x80000000))
expected = np.uint32(0x11002000)
print(f"result   = {int(result):08x}")
print(f"expected = {int(expected):08x}")
assert result == expected

# All 1s: the two rotations cancel, leaving the shifted copy
result = small_sigma0(np.uint32(0xFFFFFFFF))
expected = np.uint32(0x1FFFFFFF)
print(f"result   = {int(result):08x}")
print(f"expected = {int(expected):08x}")
assert result == expected

# Reached only if both checks pass
print("small_sigma0 tests passed")

result   = 11002000
expected = 11002000
result   = 1fffffff
expected = 1fffffff
small_sigma0 tests passed


### sigma1 (σ1)

[FIPS 180-4 equation 4.7](https://csrc.nist.gov/pubs/fips/180-4/upd1/final) 
defines this function as:

σ1(x) = ROTR^17(x) XOR ROTR^19(x) XOR SHR^10(x)

It has the same shape as σ0, with different 
numbers. Two copies of the word are rotated right by 17 and 19 places, a 
third is shifted right by 10 places, and the three results are XORed 
together. Because of the shift, the 10 bits that fall off the end are lost.

I implemented this directly by calling `rotr` and `shr` and combining the 
results with the 
[bitwise XOR operator](https://numpy.org/doc/stable/reference/generated/numpy.bitwise_xor.html) 
(`^`). I called the function `small_sigma1` to go with `big_sigma1`.

In [35]:
def small_sigma1(x):
    """
    SHA-256 sigma1 function, written as lowercase sigma 1 in the standard.

    XORs together x rotated right by 17 and 19 positions, and x shifted
    right by 10 positions (FIPS 180-4 equation 4.7).

    Parameters
    ----------
    x : numpy.uint32
        32-bit unsigned integer input.

    Returns
    -------
    numpy.uint32
        Result of ROTR(x, 17) XOR ROTR(x, 19) XOR SHR(x, 10).
    """
    # Rotate right by 17 and 19, shift right by 10, then XOR the three results
    return rotr(x, 17) ^ rotr(x, 19) ^ shr(x, 10)

The cell below shows the three copies for an input of 0x80000000. I used this instead of 1 because shifting 1 right just gives 0, so you couldn't see the shift working.

In [36]:
# A single 1 bit in the top place, so the shift shows up
x = np.uint32(0x80000000)

# Show x, each rotated or shifted copy and the XOR of the three, all in hex
print(f"x            = {int(x):08x}")
print(f"ROTR 17      = {int(rotr(x, 17)):08x}")
print(f"ROTR 19      = {int(rotr(x, 19)):08x}")
print(f"SHR 10       = {int(shr(x, 10)):08x}")
print(f"XOR of three = {int(small_sigma1(x)):08x}")

x            = 80000000
ROTR 17      = 00004000
ROTR 19      = 00001000
SHR 10       = 00200000
XOR of three = 00205000


For an input of 0x80000000, the three copies above are 00004000, 00001000 and 00200000. XORing them gives 00205000. For all 1s, the two rotated copies cancel again, leaving the shifted copy, which is 003FFFFF.

In [37]:
# A single 1 bit in the top place, with the answer worked out by hand
result = small_sigma1(np.uint32(0x80000000))
expected = np.uint32(0x00205000)
print(f"result   = {int(result):08x}")
print(f"expected = {int(expected):08x}")
assert result == expected

# All 1s: the two rotations cancel, leaving the shifted copy
result = small_sigma1(np.uint32(0xFFFFFFFF))
expected = np.uint32(0x003FFFFF)
print(f"result   = {int(result):08x}")
print(f"expected = {int(expected):08x}")
assert result == expected

# Reached only if both checks pass
print("small_sigma1 tests passed")

result   = 00205000
expected = 00205000
result   = 003fffff
expected = 003fffff
small_sigma1 tests passed


## Problem 3: Generating the SHA-256 Constants

[FIPS 180-4 section 4.2.2](https://csrc.nist.gov/pubs/fips/180-4/upd1/final) says the 64 round constants are the first thirty-two bits of the fractional parts of the cube roots of the first sixty-four prime numbers. To generate them I need to get the first 64 primes, take their cube roots, keep the fractional part of each, convert it to a 32-bit unsigned integer, show the results in hex and compare them with the standard. This first part is the primes.

### Generating the primes

The SHA-256 constants are made from the first 64 prime numbers, so first I need a function `primes(n)` that gives me the first `n` primes.

To check if a number is prime, I try dividing it by the primes I've already found. If none of them divide it exactly, it's prime. This method is called trial division ([Wikipedia: Trial division](https://en.wikipedia.org/wiki/Trial_division), [Prime Pages: trial division](https://t5k.org/glossary/xpage/TrialDivision.html)).

I only need to check primes up to the square root of the number. Factors come in pairs, so if a number had a factor bigger than its square root, it would also have a smaller one that I'd have already found ([The Math Doctors: Checking for a Prime](https://www.themathdoctors.org/?p=9893)).

As soon as one prime divides the number I stop checking it, using `break` to leave the loop early ([Python docs: break and continue statements](https://docs.python.org/3/tutorial/controlflow.html#break-and-continue-statements)).

In [38]:
def primes(n):
    """
    Generate the first n prime numbers.

    Parameters
    ----------
    n : int
        How many primes to generate, n >= 0.

    Returns
    -------
    list of int
        The first n primes in increasing order, starting at 2.
    """
    # The primes found so far
    found = []

    # The next number to test
    candidate = 2

    # Keep going until we have n primes
    while len(found) < n:
        # Assume it is prime until a smaller prime divides it
        is_prime = True

        # Try each prime found so far
        for p in found:
            # No need to go past the square root of the candidate
            if p * p > candidate:
                break

            # A prime divides it exactly, so it isn't prime
            if candidate % p == 0:
                is_prime = False
                break

        # Keep the candidate if it is prime
        if is_prime:
            found.append(candidate)

        # Move on to the next number
        candidate += 1

    return found

To test `primes`, I check the first ten primes, which I know are 2, 3, 5, 7, 11, 13, 17, 19, 23 and 29. I also check two edge cases, `n = 0` (an empty list) and `n = 1` (just 2). Then I check that `primes(64)` gives 64 primes and that the last one is 311.

In [39]:
# The first ten primes, known from any list of primes
assert primes(10) == [2, 3, 5, 7, 11, 13, 17, 19, 23, 29]

# Asking for none gives an empty list
assert primes(0) == []

# Asking for one gives just 2
assert primes(1) == [2]

# The 64th prime is 311, so there should be 64 primes ending in 311
first_64 = primes(64)
assert len(first_64) == 64
assert first_64[-1] == 311

print("primes tests passed")

primes tests passed


Now I print the first 64 primes so I can see them. The last one is 311.

In [40]:
# Show the first 64 primes
print(primes(64))

[2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41, 43, 47, 53, 59, 61, 67, 71, 73, 79, 83, 89, 97, 101, 103, 107, 109, 113, 127, 131, 137, 139, 149, 151, 157, 163, 167, 173, 179, 181, 191, 193, 197, 199, 211, 223, 227, 229, 233, 239, 241, 251, 257, 263, 269, 271, 277, 281, 283, 293, 307, 311]


### Cube roots of the first 64 primes

The standard builds each round constant from the cube root of one of the first 64 primes. It uses the first 32 bits of the fractional part of that cube root ([FIPS 180-4, section 4.2.2](https://csrc.nist.gov/pubs/fips/180-4/upd1/final), explained in simpler terms in [this HackerNoon walkthrough of SHA-256](https://hackernoon.com/a-step-by-step-guide-to-how-sha-256-works)).

The first thing I need is the cube root of each prime. `np.cbrt` does this for a whole array in one go ([numpy.cbrt](https://numpy.org/doc/stable/reference/generated/numpy.cbrt.html)). I convert the primes to floats first, because the cube roots are decimal numbers.

In [41]:
# The first 64 primes as floats, ready for the cube root
first_64_primes = np.array(primes(64), dtype=np.float64)

# The cube root of each prime
cube_roots = np.cbrt(first_64_primes)

# Show the first four cube roots
print(cube_roots[:4])

[1.25992105 1.44224957 1.70997595 1.91293118]


To check the answers, I worked one out by hand. The first prime is 2, so its cube root is the number that gives 2 when I multiply it by itself three times.

1.26:
- 1.26 × 1.26 = 1.5876
- 1.5876 × 1.26 = 2.000376

That is a bit too big. 
 
1.2599:
- 1.2599 × 1.2599 = 1.58734801
- 1.58734801 × 1.2599 = 1.999899757799

That is a bit too small. So the cube root of 2 is between 1.2599 and 1.26, which matches the first value above (1.25992105).

In [42]:
# There should be one cube root for each of the 64 primes.
assert cube_roots.shape == (64,)

# Cubing the roots should give the primes back.
assert np.allclose(cube_roots ** 3, first_64_primes)

# The first root should be about 1.26, as checked by hand.
assert np.isclose(cube_roots[0], 1.26, atol=0.001)

# The first root should be between 1.2599 and 1.26, as checked by hand
assert 1.2599 < cube_roots[0] < 1.26

print("cube root tests passed")

cube root tests passed


### Fractional parts of the cube roots

The standard only keeps the part after the decimal point of each cube root, which is called the fractional part. For example, the cube root of 2 is 1.2599…, so its fractional part is 0.2599….

To get it, I take away the whole number part of each cube root. `np.floor` rounds a number down to the nearest whole number ([numpy.floor](https://numpy.org/doc/stable/reference/generated/numpy.floor.html)), so a cube root minus its floor leaves just the decimals.

In [43]:
# The fractional part is the cube root minus its whole number part
fractions = cube_roots - np.floor(cube_roots)

# Show the first four fractional parts
print(fractions[:4])

[0.25992105 0.44224957 0.70997595 0.91293118]


To check the answers, I use the hand calculation from before. The cube root of 2 is between 1.2599 and 1.26, and its whole number part is 1. Taking 1 away gives 1.2599 − 1 = 0.2599 and 1.26 − 1 = 0.26, so the fractional part should be between 0.2599 and 0.26. That matches the first value above (0.25992105). I also check that every fractional part is at least 0 and less than 1.

In [44]:
# There should be one fractional part for each of the 64 primes
assert fractions.shape == (64,)

# Every fractional part should be at least 0 and less than 1
assert np.all(fractions >= 0) and np.all(fractions < 1)

# The first one should be between 0.2599 and 0.26
assert 0.2599 < fractions[0] < 0.26

print("fractional part tests passed")

fractional part tests passed


### Converting the fractions to 32-bit integers

Each fractional part is a decimal between 0 and 1, but the constants need to be 32-bit whole numbers. To turn a fraction into binary, you keep multiplying it by 2. Each time, the whole number part (0 or 1) is the next binary digit, and you carry on with what's left ([TutorChase: binary equivalent of a decimal fraction](https://www.tutorchase.com/answers/a-level/computer-science/how-do-you-determine-the-binary-equivalent-of-a-decimal-fraction), [GeeksforGeeks: convert a real number to a binary string](https://www.geeksforgeeks.org/?p=402168)).

For example, with 0.375:
- 0.375 × 2 = 0.75, so the digit is 0.
- 0.75 × 2 = 1.5, so the digit is 1.
- 0.5 × 2 = 1.0, so the digit is 1.

So 0.375 is .011 in binary.

I need the first 32 digits of each fraction. Doubling 32 times is the same as multiplying by 2^32 (4,294,967,296) once and keeping the whole number part. With the example, 0.375 × 2^3 = 3, which is 011 in binary, the same digits.

So I multiply each fraction by 2^32 and round down with `np.floor` to get rid of the decimals. Then I store the result as `np.uint32` ([NumPy data types](https://numpy.org/doc/stable/user/basics.types.html), which shows `.astype()` converting an array to another type). Each fraction is less than 1, so each result is less than 2^32 and fits in 32 bits.

In [45]:
# Scale each fraction up by 2**32 so its first 32 binary digits become a whole number
scaled = fractions * 2**32

# Round down to a whole number and store it as a 32-bit unsigned integer
constants = np.floor(scaled).astype(np.uint32)

# Show the first four constants
print(constants[:4])

[1116352408 1899447441 3049323471 3921009573]


To check the answers, I worked out the limits by hand. Earlier I showed that the first fraction is between 0.2599 and 0.26. The constant is the fraction multiplied by 2^32 (4,294,967,296), so I multiplied both limits by that:

- 0.2599 × 4,294,967,296 = 1,116,262,000.2304, which is the lowest the first constant can be.
- 0.26 × 4,294,967,296 = 1,116,691,496.96, which is the highest it can be.

So the first constant should be somewhere between about 1,116,262,000 and 1,116,691,496. The first value above is 1116352408, which is inside this range.

In [46]:
# There should be one constant for each of the 64 primes
assert constants.shape == (64,)

# They should be stored as 32-bit unsigned integers
assert constants.dtype == np.uint32

# The first constant should be between the two limits worked out by hand
assert 0.2599 * 2**32 < constants[0] < 0.26 * 2**32

print("uint32 conversion tests passed")

uint32 conversion tests passed


### Displaying the constants in hexadecimal

The standard writes the constants in hexadecimal, so I convert mine to match. Hex is easier to read than a long decimal number. Each hex digit stands for 4 bits, so a 32-bit number is always exactly 8 hex digits.

I use an f-string with the format `08x`. The `x` means lowercase hex, and the `08` means pad with zeros on the left until it is 8 digits long, so a small number like 255 still shows as `000000ff` ([Python Morsels: f-string cheat sheet](https://www.pythonmorsels.com/string-formatting/), [Finxter: f-string hex](https://blog.finxter.com/?p=1248792)).

In [47]:
# Format each constant as 8 hex digits, with zeros on the left if needed
hex_constants = [f"{c:08x}" for c in constants]

Now I print the constants eight to a row, the same way the standard lists them, so they are easy to compare.

In [48]:
# Print the constants eight to a row
for i in range(0, 64, 8):
    print(" ".join(hex_constants[i:i + 8]))

428a2f98 71374491 b5c0fbcf e9b5dba5 3956c25b 59f111f1 923f82a4 ab1c5ed5
d807aa98 12835b01 243185be 550c7dc3 72be5d74 80deb1fe 9bdc06a7 c19bf174
e49b69c1 efbe4786 0fc19dc6 240ca1cc 2de92c6f 4a7484aa 5cb0a9dc 76f988da
983e5152 a831c66d b00327c8 bf597fc7 c6e00bf3 d5a79147 06ca6351 14292967
27b70a85 2e1b2138 4d2c6dfc 53380d13 650a7354 766a0abb 81c2c92e 92722c85
a2bfe8a1 a81a664b c24b8b70 c76c51a3 d192e819 d6990624 f40e3585 106aa070
19a4c116 1e376c08 2748774c 34b0bcb5 391c0cb3 4ed8aa4a 5b9cca4f 682e6ff3
748f82ee 78a5636f 84c87814 8cc70208 90befffa a4506ceb bef9a3f7 c67178f2


To check the answers, I converted the first constant (1,116,352,408) to hex by hand. You divide by 16 again and again, write down each remainder as a hex digit (10 = a, 15 = f), and stop when you reach 0. We did this with Etain in first year.

- 1,116,352,408 ÷ 16 = 69,772,025 remainder 8
- 69,772,025 ÷ 16 = 4,360,751 remainder 9
- 4,360,751 ÷ 16 = 272,546 remainder 15 (f)
- 272,546 ÷ 16 = 17,034 remainder 2
- 17,034 ÷ 16 = 1,064 remainder 10 (a)
- 1,064 ÷ 16 = 66 remainder 8
- 66 ÷ 16 = 4 remainder 2
- 4 ÷ 16 = 0 remainder 4

Reading the remainders from last to first gives 4, 2, 8, a, 2, f, 9, 8, so the first constant is `428a2f98`. This matches the first value above.

In [49]:
# There should be one hex string for each of the 64 constants
assert len(hex_constants) == 64

# Every string should be exactly 8 hex digits long
assert all(len(h) == 8 for h in hex_constants)

# Leading zeros are kept, so 255 shows as 000000ff
assert f"{255:08x}" == "000000ff"

# The first constant should match the hand calculation
assert hex_constants[0] == "428a2f98"

print("hex display tests passed")

hex display tests passed


### Comparing with the standard

The last step is to check my constants against the ones in the Secure Hash Standard. Section 4.2.2 lists all 64 of them in hex ([FIPS 180-4, section 4.2.2](https://csrc.nist.gov/pubs/fips/180-4/upd1/final)).

I copied them in exactly as the standard shows them, eight to a row, as one block of text. Then `.split()` turns it into a list of 64 strings. With nothing inside the brackets, `.split()` splits on any spaces or new lines, so I don't have to separate the values myself.

In [50]:
# The 64 round constants copied from section 4.2.2 of the standard
standard_text = """
    428a2f98 71374491 b5c0fbcf e9b5dba5 3956c25b 59f111f1 923f82a4 ab1c5ed5
    d807aa98 12835b01 243185be 550c7dc3 72be5d74 80deb1fe 9bdc06a7 c19bf174
    e49b69c1 efbe4786 0fc19dc6 240ca1cc 2de92c6f 4a7484aa 5cb0a9dc 76f988da
    983e5152 a831c66d b00327c8 bf597fc7 c6e00bf3 d5a79147 06ca6351 14292967
    27b70a85 2e1b2138 4d2c6dfc 53380d13 650a7354 766a0abb 81c2c92e 92722c85
    a2bfe8a1 a81a664b c24b8b70 c76c51a3 d192e819 d6990624 f40e3585 106aa070
    19a4c116 1e376c08 2748774c 34b0bcb5 391c0cb3 4ed8aa4a 5b9cca4f 682e6ff3
    748f82ee 78a5636f 84c87814 8cc70208 90befffa a4506ceb bef9a3f7 c67178f2
"""

# Split the text on whitespace to get a list of 64 hex strings
standard_constants = standard_text.split()

Now I check whether my list is exactly the same as the standard's list.

In [51]:
# Check whether my list is exactly the same as the standard's list
print(hex_constants == standard_constants)

True


When `==` is used on two lists, it is only `True` if they have the same items in the same order, so one wrong constant would make it `False`. For the tests, I assert that the two lists are equal. I also change one value in a copy of my list and check that the comparison now fails. This shows the check can catch a mistake.

In [52]:
# The standard should have 64 constants
assert len(standard_constants) == 64

# All of my constants should match the standard's
assert hex_constants == standard_constants

# Change one value in a copy of my list
changed = hex_constants.copy()
changed[0] = "00000000"

# The comparison should now fail
assert changed != standard_constants

print("all 64 constants match the standard")

all 64 constants match the standard


All 64 of my constants match the standard, so the method works: take the cube root of each prime, keep the fractional part, multiply by 2^32 and round down. A forum post also generates these constants in Zig using 64-bit floats ([Ziggit: generating the SHA256 magic constants](https://ziggit.dev/t/generating-the-sha256-magic-constants-using-comptime/11705)). My results matching the standard shows that 64-bit floats are precise enough for the 32 bits needed here.

## Problem 4: Padding and Parsing Messages

### Padding the message

SHA-256 works on blocks of exactly 512 bits, but most messages aren't that long, so the message is padded first. Section 5.1.1 of the standard explains how ([FIPS 180-4, section 5.1.1](https://csrc.nist.gov/pubs/fips/180-4/upd1/final)), and [Axiom Academy's SHA-256 lesson](https://theaxiomacademy.org/learn/crypto-lesson-sha256) lists the same steps:

1. Add a single 1 bit to the end of the message.
2. Add k zero bits.
3. Add the length of the original message in bits, as a 64-bit number.

The standard says k is the smallest number (0 or more) where l + 1 + k = 448 (mod 512). Here l is the length of the message in bits, and "mod 512" just means the remainder after dividing by 512. This makes the message end 64 bits short of a full block, so the 64-bit length fits in the gap.

Rearranging gives k = 447 − l (mod 512), which is what the function below calculates.

In [53]:
def padding_zero_bits(length_bits):
    """
    Work out how many zero bits the padding needs (FIPS 180-4 section 5.1.1).

    Parameters
    ----------
    length_bits : int
        The length of the message in bits.

    Returns
    -------
    int
        The smallest k, with k >= 0, where length_bits + 1 + k is 448 mod 512.
    """
    # Rearranging l + 1 + k = 448 (mod 512) gives k = 447 - l (mod 512)
    return (447 - length_bits) % 512

The `%` in the function gives the remainder after dividing. If 447 − l is negative, which happens when l is 448 or more, Python's `%` still gives an answer from 0 to 511 ([DataCamp: modulo operator in Python](https://www.datacamp.com/tutorial/modulo-operator-python)). That is the smallest k that isn't negative, which is what I need. Here is an example with l = 448:

In [54]:
# 447 - 448 is negative, because a 448 bit message has no room left
print(447 - 448)

# Python's % still gives a result from 0 to 511
print((447 - 448) % 512)

-1
511


To test the function, I started with the standard's own example. "abc" is 3 bytes, which is 24 bits. The standard says it gets a 1 bit, then 423 zero bits, then the 64-bit length. 24 + 1 + 423 + 64 = 512, so it is exactly one block.

I also worked out some other lengths by hand:

- An empty message (l = 0) needs 447 zero bits, because 0 + 1 + 447 + 64 = 512.
- l = 447 needs 0 zero bits, because 447 + 1 + 0 + 64 = 512. The message only just fits.
- l = 448 needs 511 zero bits. There is no room for the length after the 1 bit, so the padding spills into a second block: 448 + 1 + 511 + 64 = 1024.
- l = 512 (one full block) needs 447 zero bits, because 512 + 1 + 447 + 64 = 1024.

In [55]:
# The standard's own example: "abc" is 24 bits and needs 423 zero bits
assert padding_zero_bits(24) == 423

# An empty message needs 447 zero bits
assert padding_zero_bits(0) == 447

# 447 bits only just fits, so no zero bits are needed
assert padding_zero_bits(447) == 0

# 448 bits spills into a second block, so 511 zero bits are needed
assert padding_zero_bits(448) == 511

# One full block of 512 bits needs 447 zero bits
assert padding_zero_bits(512) == 447

print("padding zero bits tests passed")

padding zero bits tests passed


Last, I use a loop to check the rule from the standard for every length from 0 to 1999 bits. `range(2000)` gives the numbers 0 up to 1999.

In [56]:
# Go through every message length from 0 to 1999 bits
for length_bits in range(2000):
    # Work out the number of zero bits for this length
    k = padding_zero_bits(length_bits)

    # k should never be negative and never 512 or more
    assert 0 <= k < 512

    # The message, the 1 bit and the zeros should end 64 bits short of a block
    assert (length_bits + 1 + k) % 512 == 448

print("padding loop test passed")

padding loop test passed


### Building the padded message

Now I use `padding_zero_bits` to pad a whole message. A message is always a whole number of bytes, so the 1 bit and the 7 zero bits after it make one byte: `10000000`, which is `0x80`. The rest of the zero bits are whole zero bytes, and the length goes at the end as 8 bytes (big-endian, like in Problem 1). The steps are the same as in [FIPS 180-4, section 5.1.1](https://csrc.nist.gov/pubs/fips/180-4/upd1/final).

In [57]:
def pad_message(msg):
    """
    Pad a message as described in FIPS 180-4 section 5.1.1.

    Parameters
    ----------
    msg : bytes
        The message to pad.

    Returns
    -------
    bytes
        The padded message, whose length is a multiple of 64 bytes.
    """
    # The length of the message in bits
    length_bits = len(msg) * 8

    # The number of zero bits needed after the 1 bit
    k = padding_zero_bits(length_bits)

    # The 1 bit followed by 7 zero bits makes one byte
    one_bit = b"\x80"

    # The other zero bits as whole zero bytes
    zeros = bytes((k - 7) // 8)

    # The length of the message as 8 bytes, big-endian
    length = length_bits.to_bytes(8, byteorder="big")

    # Put the pieces together
    return msg + one_bit + zeros + length

Some lines in the function need explaining:

- `0x80` is `10000000` in binary, so the 1 bit and the next 7 zero bits are one byte.
- `k` counts the zero bits, and 7 of them are already in that `0x80` byte. So I take off 7 and divide by 8 to get the number of whole zero bytes. `//` divides and drops anything after the decimal point, but here it always divides exactly.
- `bytes(n)` makes `n` bytes that are all zero.
- `to_bytes(8, byteorder="big")` turns a number into 8 bytes with the biggest byte first, like I did in Problem 1.

Here are those last two on their own:

In [58]:
# bytes(3) makes three zero bytes
print(bytes(3))

# 24 as 8 bytes, big-endian, so the 24 (0x18) is in the last byte
print((24).to_bytes(8, byteorder="big"))

b'\x00\x00\x00'
b'\x00\x00\x00\x00\x00\x00\x00\x18'


To test it, I start with the "abc" example from the standard. It shows the bits of a, b and c, then a 1, then zeros, then the length (24) in binary. In hex that is `616263`, then `80`, then 52 zero bytes, then the length written as 8 bytes, `0000000000000018`. That is 3 + 1 + 52 + 8 = 64 bytes, so one block.

I also check some lengths by hand:

- An empty message is just the `80` byte, zeros and the length, so it makes 64 bytes.
- 55 bytes is 440 bits. It is the longest message that still fits in one block, because 440 + 1 + 7 + 64 = 512.
- 56 bytes is 448 bits, which has no room for the length, so it needs two blocks (128 bytes).
- 64 bytes is a full block already, so the padding adds a whole extra block (128 bytes).

In [59]:
# The padded "abc" from the standard, written in hex
expected_abc = "616263" + "80" + "00" * 52 + "0000000000000018"

# The padded message for "abc" should match it
assert pad_message(b"abc").hex() == expected_abc

# An empty message should be padded to one block of 64 bytes
assert len(pad_message(b"")) == 64

# 55 bytes is the longest message that fits in one block
assert len(pad_message(bytes(55))) == 64

# 56 bytes has no room left for the length, so it needs two blocks
assert len(pad_message(bytes(56))) == 128

# 64 bytes is a full block, so the padding adds a whole extra block
assert len(pad_message(bytes(64))) == 128

print("pad message tests passed")

pad message tests passed


### Splitting a block into words

SHA-256 reads the padded message one 512-bit block at a time, and each block is sixteen 32-bit words ([FIPS 180-4, section 5.2.1](https://csrc.nist.gov/pubs/fips/180-4/upd1/final)). I already did this for one block in Problem 1, so here I turn that code into a function. I changed three things:

- It is a function now, so I can use it on any block.
- The block is a parameter, instead of a variable I set earlier.
- I put the list comprehension on separate lines, so no line is longer than 79 characters ([PEP 8: maximum line length](https://peps.python.org/pep-0008/#maximum-line-length)).

In [60]:
def block_to_words(block):
    """
    Split a 64-byte block into sixteen 32-bit words (FIPS 180-4 section 5.2.1).

    Parameters
    ----------
    block : bytes
        One 512-bit block, which is 64 bytes.

    Returns
    -------
    numpy.ndarray
        Sixteen words, as 32-bit unsigned integers.
    """
    # Read each group of four bytes as one big-endian integer
    words = [
        int.from_bytes(block[i:i + 4], byteorder="big")
        for i in range(0, 64, 4)
    ]

    # Store the words as 32-bit unsigned integers
    return np.array(words, dtype=np.uint32)

To test it, I use the block from Problem 1, which has the bytes 0 to 63 in order. The first word is bytes 0 to 3, which is `0x00010203` or 66051. The last word is bytes 60 to 63. In hex those bytes are 3c, 3d, 3e and 3f, so the word is `0x3c3d3e3f`, which is 1010646591.

I also use the padded "abc" from the standard. Its first word is the bytes of a, b and c plus the `0x80` byte, so `0x61626380`. Its last word is the length, 24, which is `0x18`. All the words in between should be zero.

In [61]:
# The block from Problem 1 has the bytes 0 to 63 in order
test_words = block_to_words(bytes(range(64)))

# There should be sixteen words
assert len(test_words) == 16

# Each word should be a 32-bit unsigned integer
assert test_words.dtype == np.uint32

# The first word is bytes 0 to 3, which is 0x00010203 or 66051
assert test_words[0] == 66051

# The last word is bytes 60 to 63, which is 0x3c3d3e3f or 1010646591
assert test_words[15] == 1010646591

# Pad "abc" and split it into words, like in the standard
abc_words = block_to_words(pad_message(b"abc"))

# The first word is the bytes of a, b and c, then the 0x80 byte
assert abc_words[0] == 0x61626380

# The last word is the length, 24 bits, which is 0x18
assert abc_words[15] == 0x18

# Every word in between should be zero
assert all(abc_words[1:15] == 0)

print("block to words tests passed")

block to words tests passed


## Problem 5: The SHA-256 Compression Function

## Problem 6: Complete SHA-256